# Web Scraping Falcon 9 Launch Records from Wikipedia
Goal: scrape the *List of Falcon 9 and Falcon Heavy launches* tables with BeautifulSoup and build a DataFrame.

In [1]:
import re, unicodedata, requests
import pandas as pd
from bs4 import BeautifulSoup

## 1. Request the page (fixed revision so results are reproducible)

In [2]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
headers = {'User-Agent': 'Mozilla/5.0 (Data Science Capstone project)'}
response = requests.get(static_url, headers=headers, timeout=30)
print('Status:', response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

Status: 200
List of Falcon 9 and Falcon Heavy launches - Wikipedia


## 2. Column names from the first launch table

In [3]:
html_tables = soup.find_all('table')
first_launch_table = html_tables[2]

def extract_column_from_header(row):
    if row.br: row.br.extract()
    if row.a: row.a.extract()
    if row.sup: row.sup.extract()
    colname = ' '.join(row.contents)
    if not colname.strip().isdigit():
        return colname.strip()

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## 3. Parsing helpers

In [4]:
def date_time(cell):
    return [t.strip() for t in list(cell.strings)][0:2]

def booster_version(cell):
    return ''.join([bv for i, bv in enumerate(cell.strings) if i % 2 == 0][0:-1])

def landing_status(cell):
    return [i for i in cell.strings][0]

def get_mass(cell):
    mass = unicodedata.normalize('NFKD', cell.text).strip()
    if mass:
        return mass[0:mass.find('kg') + 2]
    return 0

## 4. Extract every launch row

In [5]:
launch_dict = dict.fromkeys(column_names)
for k in ['Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']:
    launch_dict.pop(k, None)
launch_dict = {'Flight No.': [], 'Launch site': [], 'Payload': [], 'Payload mass': [], 'Orbit': [],
               'Customer': [], 'Launch outcome': [], 'Version Booster': [], 'Booster landing': [],
               'Date': [], 'Time': []}

for table in soup.find_all('table', 'wikitable plainrowheaders collapsible'):
    for rows in table.find_all('tr'):
        flag = False
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        row = rows.find_all('td')
        if flag and len(row) >= 9:
            try:
                dt = date_time(row[0])
                bv = booster_version(row[1])
                if not bv:
                    bv = row[1].a.string
                launch_dict['Flight No.'].append(flight_number)
                launch_dict['Date'].append(dt[0].strip(','))
                launch_dict['Time'].append(dt[1])
                launch_dict['Version Booster'].append(bv)
                launch_dict['Launch site'].append(row[2].a.string)
                launch_dict['Payload'].append(row[3].a.string)
                launch_dict['Payload mass'].append(get_mass(row[4]))
                launch_dict['Orbit'].append(row[5].a.string)
                launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].text.strip())
                launch_dict['Launch outcome'].append(list(row[7].strings)[0])
                launch_dict['Booster landing'].append(landing_status(row[8]))
            except (AttributeError, IndexError):
                # drop the partially appended row so all columns keep the same length
                n = min(len(v) for v in launch_dict.values())
                for k in launch_dict: launch_dict[k] = launch_dict[k][:n]
print({k: len(v) for k, v in launch_dict.items()})

{'Flight No.': 121, 'Launch site': 121, 'Payload': 121, 'Payload mass': 121, 'Orbit': 121, 'Customer': 121, 'Launch outcome': 121, 'Version Booster': 121, 'Booster landing': 121, 'Date': 121, 'Time': 121}


In [6]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.head(10)

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
5,6,VAFB,CASSIOPE,500 kg,Polar orbit,MDA,Success,F9 v1.17B1003,Uncontrolled,29 September 2013,16:00
6,7,CCAFS,SES-8,"3,170 kg",GTO,SES,Success,F9 v1.1,No,3 December 2013,22:41
7,8,CCAFS,Thaicom 6,"3,325 kg",GTO,Thaicom,Success,F9 v1.1,No,6 January 2014,22:06
8,9,Cape Canaveral,SpaceX CRS-3,"2,296 kg",LEO,NASA,Success,F9 v1.1,Controlled,18 April 2014,19:25
9,10,Cape Canaveral,Orbcomm-OG2,"1,316 kg",LEO,Orbcomm,Success,F9 v1.1,Controlled,14 July 2014,15:15


In [7]:
df.to_csv('spacex_web_scraped.csv', index=False)
print(df.shape)
df['Booster landing'].value_counts()

(121, 11)


Booster landing
Success         80
No              19
Failure         10
Controlled       5
No attempt       3
Uncontrolled     2
Failure          1
Precluded        1
Name: count, dtype: int64